# Abstract Interpretation for Neural Networks

## What This Is
Abstract interpretation is a program analysis technique that over-approximates the set of possible values a program variable can take. For neural networks:

- **Box domains (IBP)**: Tracks lower/upper bounds per neuron — fast but loose
- **Zonotopes**: Tracks linear correlations between neurons — tighter, polynomial cost
- **DeepPoly**: Uses affine over-approximations per neuron that are tighter than zonotopes on some architectures

## DeepPoly
DeepPoly (Singh et al., 2019) assigns each neuron an affine over-approximation:
- Upper bound: hᵢ ≤ αᵢᵀx + cᵢ (affine function of the *original* inputs)
- Lower bound: hᵢ ≥ αᵢ'ᵀx + cᵢ'

These bounds are threaded back to the input layer for the final output bounds. DeepPoly handles ReLU, MaxPool, and other piecewise-linear operations.

For ReLU with ambiguous neuron (lo < 0 < hi):
- Upper bound: slope × pre + intercept (tangent line, same as CROWN)
- Lower bound: choose slope ∈ {0, 1} based on which gives tighter bound

In [ ]:
import numpy as np
from typing import List, Dict, Tuple

np.random.seed(42)

# DeepPoly-style bound propagation
# Each neuron maintains (lower_affine_coefs, upper_affine_coefs)
# where coefs are w.r.t. the original input x

class DeepPolyAbstractDomain:
    """
    Simplified DeepPoly: tracks affine lower/upper bounds per neuron
    w.r.t. the *original* input x.
    """
    
    def __init__(self, x0: np.ndarray, eps: float):
        n = len(x0)
        self.x0 = x0
        self.eps = eps
        # Input constraints: x0 - eps <= x <= x0 + eps
        # Represent each input neuron as (e_i, -e_i) meaning [x0[i]-eps, x0[i]+eps]
        self.x_lo = x0 - eps
        self.x_hi = x0 + eps
        
        # Current layer bounds (concrete, not affine for simplicity)
        self.h_lo = self.x_lo.copy()
        self.h_hi = self.x_hi.copy()
    
    def linear_layer(self, W: np.ndarray, b: np.ndarray) -> 'DeepPolyAbstractDomain':
        """Apply linear transformation W @ h + b."""
        Wp = np.maximum(0, W); Wn = np.minimum(0, W)
        new_lo = Wp @ self.h_lo + Wn @ self.h_hi + b
        new_hi = Wp @ self.h_hi + Wn @ self.h_lo + b
        result = DeepPolyAbstractDomain.__new__(DeepPolyAbstractDomain)
        result.x0, result.eps, result.x_lo, result.x_hi = self.x0, self.eps, self.x_lo, self.x_hi
        result.h_lo, result.h_hi = new_lo, new_hi
        return result
    
    def relu_layer(self) -> 'DeepPolyAbstractDomain':
        """Apply ReLU with DeepPoly-style relaxation."""
        n = len(self.h_lo)
        new_lo = np.zeros(n)
        new_hi = np.zeros(n)
        
        for i in range(n):
            lo, hi = self.h_lo[i], self.h_hi[i]
            if lo >= 0:
                new_lo[i] = lo
                new_hi[i] = hi
            elif hi <= 0:
                new_lo[i] = 0.0
                new_hi[i] = 0.0
            else:
                # DeepPoly lower bound slope: max(0, min(1, ...)) — use 0 when |lo|<hi
                # Heuristic: use 0 when |lo| > hi (neuron probably inactive)
                slope = 1.0 if hi > abs(lo) else 0.0
                new_lo[i] = max(0, slope * lo)
                # Upper bound: tangent line
                upper_slope = hi / (hi - lo)
                new_hi[i] = upper_slope * hi - upper_slope * lo
        
        result = DeepPolyAbstractDomain.__new__(DeepPolyAbstractDomain)
        result.x0, result.eps, result.x_lo, result.x_hi = self.x0, self.eps, self.x_lo, self.x_hi
        result.h_lo, result.h_hi = new_lo, new_hi
        return result

# Test DeepPoly on a small network
W1 = np.array([[2.0, 1.0], [-1.0, 2.0], [1.5, -0.5]])
b1 = np.array([0.5, 0.5, 0.2])
W2 = np.array([[1.0, 0.5, 0.3], [-1.0, -0.5, -0.3]])
b2 = np.array([0.1, -0.1])

x0 = np.array([0.5, 0.3])

print('DeepPoly Bound Comparison vs IBP')
print('='*50)

for eps in [0.05, 0.1, 0.2]:
    # DeepPoly
    dp = DeepPolyAbstractDomain(x0, eps)
    dp = dp.linear_layer(W1, b1)
    dp = dp.relu_layer()
    dp = dp.linear_layer(W2, b2)
    dp_gap = (dp.h_hi[0] - dp.h_lo[0])
    
    # IBP
    xlo, xhi = x0-eps, x0+eps
    W1p, W1n = np.maximum(0,W1), np.minimum(0,W1)
    p_lo = W1p@xlo + W1n@xhi + b1
    p_hi = W1p@xhi + W1n@xlo + b1
    h_lo, h_hi = np.maximum(0,p_lo), np.maximum(0,p_hi)
    W2p, W2n = np.maximum(0,W2), np.minimum(0,W2)
    ibp_lo = W2p@h_lo + W2n@h_hi + b2
    ibp_hi = W2p@h_hi + W2n@h_lo + b2
    ibp_gap = ibp_hi[0] - ibp_lo[0]
    
    print(f'eps={eps}: IBP gap={ibp_gap:.4f}, DeepPoly gap={dp_gap:.4f}, '
          f'improvement={max(0, (ibp_gap-dp_gap)/ibp_gap*100):.1f}%')
